# 원본/PQ × 동일 FIQA 보정 교차 대조

첫 셀의 `LFW_PROTOCOL_MODE`로 실행 경로를 선택합니다. 현재 기본값은 `blufr_calibration`입니다.
공개 BLUFR train/test 분할을 유지하며 독립 보정을 수행합니다. 별도 benchmark 평가는 실행하지 않습니다.
Origin+PQ m128/m64/m32에 같은 세 가지 보정을 적용하고 순위 실패와 threshold 실패를 구분합니다.

**Kernel Restart → Run All**. 먼저 `EXECUTE=False`로 입력을 점검하고 `ready=True`일 때 실행합니다.
새 경로의 전체 행렬은 4모델×10 trials×7비율×20 seeds = 5,600 jobs입니다.
`MAX_NEW_JOBS_PER_RUN=None`은 전체 미완료 작업, 양의 정수는 이번 호출에서 추가할 작업 수입니다.
`DISPLAY_*`는 출력 표만 선택합니다. 완료 작업은 SQLite에서 재사용하고 PQ/threshold는 새 분할에서 적합합니다.
`legacy`는 기존 3-dataset 실험이며 새 경로와 결과를 합치지 않습니다.
메모리 점검은 단계 사이 점검이며 강제 메모리 상한이나 OS 장애 방지 보장은 아닙니다.


### BLUFR 공개 분할 기반의 독립 보정

공개 train/test·test gallery 1,000명·probe 목록을 그대로 사용합니다. train 1,500명 내부에서
압축 학습:보정 = 5:5, 6:4, 7:3, 8:2, 4:6, 3:7, 2:8을 비교합니다.
calibration gallery는 고정 100명×1장입니다. 100→1,000명 검색 규모 전이이며, gallery 이미지 선택 방식도 다릅니다.
별도 BLUFR benchmark 평가는 실행하지 않습니다. 기존 benchmark Python 코드는 보존합니다.
설정 변경 후 **Kernel Restart → Run All**. 기존 완료 run은 덮어쓰지 않습니다.

주 실행점입니다. 처음에는 EXECUTE=False로 검사하고 ready=True일 때 True로 실행합니다.
YAML의 10 trials×7비율×4모델×20 fit/safety seeds = 5,600 jobs입니다. 각 job은 Origin+3PQ를 함께 비교합니다.
DATASETS/RUN_MATRIX/REUSE_PQ_MODELS/SOURCE_REPORT_DIR는 새 경로에 적용되지 않습니다. PQ는 개발 인물만으로 새로 학습합니다.
DISPLAY_*는 표시 필터입니다. 같은 설정으로 재실행하면 완료 작업을 재사용합니다.


In [ ]:
from __future__ import annotations

# LFW 별도 실험 선택. legacy는 기존 실행 설정과 동작을 유지합니다.
LFW_PROTOCOL_MODE = "blufr_calibration"  # "legacy" / "matched_calibration" / "blufr_calibration"
LFW_CALIBRATION_CONFIG_PATH = "configs/experiments/lfw_blufr_calibration.yaml"
LFW_MATCHED_OUTPUT_ROOT = "results/calibration/lfw_matched"
if LFW_PROTOCOL_MODE not in ("legacy", "matched_calibration", "blufr_calibration"):
    raise ValueError("이 노트북에서 지원하지 않는 LFW_PROTOCOL_MODE입니다.")

# 0. 모든 사용자 설정 — 변경 후 Kernel Restart → Run All
from pathlib import Path
import sys
from copy import deepcopy

PROJECT_ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
                     if (p / "research").is_dir() and (p / ".git").exists()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("프로젝트 안에서 실행하세요.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from research.experiments.calibration_matrix import DEFAULT_RUN_MATRIX
RUN_MATRIX = deepcopy(DEFAULT_RUN_MATRIX)  # 명시적 기존 12개 run; 자동 latest 선택 없음

DATASETS = ("lfw", "rfw_custom", "survface")
MODELS = ("arcface", "adaface", "magface", "edgeface")
PQ_PROFILES = ("pq_512_m128_b8", "pq_512_m64_b8", "pq_512_m32_b8")
FIQA_VARIANT = "L"
CALIBRATION_ROOT = PROJECT_ROOT / "results/calibration"
OUTPUT_ROOT = CALIBRATION_ROOT / "origin_vs_pq"

EXECUTE = False
PARTITION_SEEDS = (*range(19), 8972)
ORIGIN_SHARD_SIZE = 4096

# 전체 실험 범위는 유지하며 이번 실행에서 새로 계산할 run/seed 작업만 제한합니다.
MAX_NEW_JOBS_PER_RUN = None  # None이면 전체 미완료 작업
BLAS_THREADS = 2
MIN_AVAILABLE_RAM_GB = 8.0
MAX_PROCESS_RAM_GB = 16.0  # 단계 사이 점검값이며 강제 메모리 상한이 아닙니다.
EXPORT_CHAT = True

# 기존 PQ 적합 계수 재사용: source·입력·fit 설정·과학 코드·runtime 검증 후 허용.
# 불일치하면 중단합니다. 의도적으로 새 설정을 쓰려면 False로 바꾸어 새로 적합하세요.
REUSE_PQ_MODELS = True
SOURCE_REPORT_DIR = CALIBRATION_ROOT / "matrix/reports/matrix-report-d690567c6f4f7e7ad44ef19e"
FIT_SETTINGS = dict(
    target_fpirs=(.01, .05, .10, .20, .30),
    safety_fraction=.30, minimum_group_non_mated=100, shrinkage_strength=200.,
    knot_quantiles=(1/3, 2/3), smoothing=.01, ridge=.001, max_iterations=2000,
    margin_slope_cap=.95,  # 기존 quality-only 모델 기록과 일치; FIQA 단조성 제약 아님
    resamples=2000, bootstrap_seed=8972,
    diagnostic_fpir_grid=(0., .001, .005, .01, .02, .05, .1, .2, .3, .5, 1.),
)

# 아래는 표시할 표만 선택하며 실험 범위/방법 선택에 사용하지 않습니다.
DISPLAY_DATASET = "lfw"
DISPLAY_MODEL = "arcface"
DISPLAY_TARGET_FPIR = .10
DISPLAY_DIAGNOSTIC_FPIR = .10

DISPLAY_TRIAL_ID = 1
DISPLAY_DEVELOPMENT_FRACTION = .5


## 1. 입력과 실행 범위 확인

기본 범위는 3개 데이터셋 × 4모델 × (원본 + 3개 PQ) = **48개 표현 조건**입니다.
48 × 3방법 × 5목표 × 20분할 = **14,400개 운영 성능 행**을 저장합니다.
범위를 바꾸면 아래에서 실제 행 수를 다시 계산합니다.

원본 test의 최대 점수·정답 점수·정답 순위는 기존 검색 ledger를 재사용합니다.
원본 calibration만 동일한 enrollment 규칙으로 정확 cosine 검색합니다.
이 검색은 기존 NumPy **CPU** 구현이며, FR/FIQA 추론·Grad-CAM·압축기 학습·DB 작업은 없습니다.
원본과 PQ의 점수 공간이 다르므로 각각 보정하며 원본 threshold를 ADC로 복사하지 않습니다.

PQ condition score 또는 FIQA가 없거나 손상되면 정확한 경로와 함께 중단합니다.
누락 입력은 기존 입력 준비 절차에서 만든 뒤 재개하세요.

In [ ]:
if LFW_PROTOCOL_MODE == "legacy":
    import pandas as pd
    from IPython.display import Markdown, display
    from research.experiments.origin_pq_inputs import inspect_origin_pq_experiment
    from research.experiments.origin_vs_pq_calibration import (
        OriginPQSettings, VerifiedPQModels, run_origin_pq_campaign,
    )

    settings = OriginPQSettings(**FIT_SETTINGS)
    settings.validate()
    plan = inspect_origin_pq_experiment(
        PROJECT_ROOT, run_matrix=RUN_MATRIX, datasets=DATASETS, models=MODELS,
        profiles=PQ_PROFILES, variant=FIQA_VARIANT, calibration_root=CALIBRATION_ROOT,
    )
    source_report = VerifiedPQModels(SOURCE_REPORT_DIR) if REUSE_PQ_MODELS else None
    representation_count = len(plan) + plan.source_run_id.nunique()
    expected_rows = representation_count * 3 * len(settings.target_fpirs) * len(PARTITION_SEEDS)
    print("Python:", sys.executable)
    print(f"입력 검증 완료: {plan.source_run_id.nunique()}개 run, "
          f"{representation_count}개 표현 조건, {expected_rows:,}개 운영 성능 행")
    with pd.option_context("display.max_rows", None, "display.max_colwidth", 100):
        display(plan[["dataset_id", "model", "compression_profile", "source_run_id", "test_rows", "ready"]])
    print("출력:", OUTPUT_ROOT)
    print("실험 실행:", EXECUTE, "/ 기존 PQ 모델 재사용:", REUSE_PQ_MODELS)


## 2. 제한된 묶음 실행과 재개

분할별 수천 개 CSV 대신 `checkpoints/campaign-*.sqlite3` 한 파일에 수치·모델·출처를 저장합니다.
완료한 run/seed 작업은 transaction 단위로 재사용합니다. 중단 당시 미완료 작업만 다시 계산합니다.
최초 원본 calibration 검색은 4,096질의 단위로 재개하며, 완료된 최종 점수의 hash를 검증한 뒤
중복 replay shard만 정리합니다. 원본 run과 최종 score manifest/Parquet는 유지합니다.

`REUSE_PQ_MODELS=True`는 명시한 기존 PQ 모델 계수를 검증 후 재사용합니다.
원본 cosine과 PQ ADC는 각자의 score space에서 보정하며, 같은 calibration/test cohort를 사용합니다.
원본 입력 준비는 CPU와 RAM을 사용합니다. 메모리 점검은 단계 사이에서 수행되므로 순간 최대 사용량을 제한하지 않습니다.

메모리 점검 중단이나 실행 묶음 완료 후에는 현재까지의 **부분 결과**를 채팅용으로 저장합니다.
**전체 완료 전에는 전체 행렬의 결론을 내리지 마세요.** 같은 설정으로 다시 실행하면 이어집니다.
전체 실험 완료 시에만 상세 결과 ZIP을 한 번 더 저장합니다.


In [ ]:
if LFW_PROTOCOL_MODE == "legacy":
    result = None
    if EXECUTE:
        status = display(Markdown("입력 준비 중…"), display_id=True)
        def show_progress(event):
            status.update(Markdown("진행: " + " · ".join(f"{k}={v}" for k, v in event.items())))
        result = run_origin_pq_campaign(
            plan, OUTPUT_ROOT, partition_seeds=PARTITION_SEEDS, settings=settings,
            pq_model_report=SOURCE_REPORT_DIR if REUSE_PQ_MODELS else None,
            shard_size=ORIGIN_SHARD_SIZE, progress=show_progress,
            max_new_jobs=MAX_NEW_JOBS_PER_RUN, blas_threads=BLAS_THREADS,
            minimum_available_gb=MIN_AVAILABLE_RAM_GB, maximum_process_gb=MAX_PROCESS_RAM_GB,
            export_chat=EXPORT_CHAT,
        )
        status.update(Markdown(f"전체 완료={result['completed']} · "
                              f"완료 작업={result['completed_jobs']}/{result['expected_jobs']}"))
        print("이번 새 작업:", result["new_jobs"], "/ 중단 사유:", result["stop_reason"])
        print("재개용 파일:", result["checkpoint_path"])
        if result["chat_dir"] is not None:
            print("GPT 채팅 안내:", result["chat_dir"] / "START_HERE.md")
            print("GPT 채팅 첨부:", result["chat_dir"] / "analysis.zip")
        if result["report_dir"] is not None:
            print("전체 상세 결과:", result["report_dir"] / "results.zip")
        if not result["completed"]:
            print("전체 범위의 부분 결과입니다. 같은 설정으로 다시 실행하면 이어서 처리합니다.")
    else:
        print("입력 검증만 완료했습니다. 실행하려면 EXECUTE=True로 변경하세요.")


## 3. 운영 성능과 실패 원인

Threshold는 calibration에서 고정합니다. method_summary.csv에는 반올림 전 FPIR,
목표 충족, 성공 수·분모, Wilson CI, TPIR의 신원 단위 bootstrap CI가 있습니다.
아래는 선택 조건의 분할별 min/median/max와 목표 충족 횟수입니다.

등록 질의는 **성공 / 정답 Top-K 밖의 순위 실패 / Top-K 안의 threshold 실패**로 나눕니다.
세 개의 수를 합하면 등록 질의 수입니다. rank_k_ceiling은 threshold로 회복 가능한 범위의
진단 상한이며, 같은 FPIR에서 실제 달성할 수 있다는 뜻은 아닙니다.

In [ ]:
if LFW_PROTOCOL_MODE == "legacy":
    if result is not None and "split_summary" in result:
        view = result["split_summary"]
        view = view.loc[view.dataset_id.eq(DISPLAY_DATASET) & view.model.eq(DISPLAY_MODEL)
                        & view.target_fpir.eq(DISPLAY_TARGET_FPIR)]
        display(view[["compression_profile", "method", "split_count", "target_met_split_count",
                      "fpir_min", "fpir_median", "fpir_max", "tpir_min", "tpir_median", "tpir_max",
                      "rank_failure_rate", "threshold_failure_median"]])
        print("완료된 조건 전체의 수치·실패 수는 재개용 DB와 완료 시 상세 ZIP에 보관합니다.")


## 4. 압축과 보정의 관계

paired_comparisons.csv는 같은 표현의 방법 차이와 같은 방법의 PQ−원본 차이를 담습니다.
interactions.csv의 정의는 다음과 같습니다.

**I = (PQ의 FIQA−Global) − (원본의 FIQA−Global)**

운영 표의 I는 **같은 목표 FPIR**에서의 차분입니다. 실제 FPIR는 다를 수 있으므로 네 실제
FPIR와 목표 충족을 함께 확인합니다. TPIR 차분의 CI는 같은 등록 신원을 함께 재표집합니다.
인과효과·압축 특이 알고리즘·그룹 FPIR 보장을 자동으로 뜻하지 않습니다.

In [ ]:
if LFW_PROTOCOL_MODE == "legacy":
    if result is not None and "split_summary" in result:
        view = result["interactions"]
        view = view.loc[view.dataset_id.eq(DISPLAY_DATASET) & view.model.eq(DISPLAY_MODEL)
                        & view.target_fpir.eq(DISPLAY_TARGET_FPIR)]
        # 여러 seed의 범위는 기술통계이며 하나의 합친 신뢰구간이 아닙니다.
        display(view.groupby(["compression_profile", "method"]).agg(
            split_count=("partition_seed", "count"), all_four_met=("all_four_target_met", "sum"),
            pq_gain_median=("pq_gain", "median"), origin_gain_median=("origin_gain", "median"),
            interaction_min=("interaction", "min"), interaction_median=("interaction", "median"),
            interaction_max=("interaction", "max")))
        print("각 seed의 실제 FPIR와 paired CI는 상세 결과에 보존하며, 요약 ZIP에는 범위를 담습니다.")


## 5. 같은 FPIR에서의 사후 진단

한 번 적합한 함수의 score − threshold(FIQA)를 고정하고 scalar cutoff를 이동해 곡선을 만듭니다.
같은 FPIR 좌표의 TPIR는 경험적 곡선을 선형 보간한 값입니다. 동점 구간은 무작위 threshold
혼합에 해당할 수 있으며 **운영 가능한 단일 threshold를 선택한 결과가 아닙니다**.

진단 곡선과 진단 I에는 현재 CI가 없습니다. fitted_target_fpir가 다른 모델들의 점을
하나의 ROC처럼 이어 붙이지 않습니다. 아래는 설정한 하나의 적합 목표와 하나의 진단 FPIR만
표시합니다. 운영 결과는 앞의 표를 사용하세요.

In [ ]:
if LFW_PROTOCOL_MODE == "legacy":
    if result is not None and "split_summary" in result:
        view = result["diagnostic_interactions"]
        view = view.loc[view.dataset_id.eq(DISPLAY_DATASET) & view.model.eq(DISPLAY_MODEL)
                        & view.fitted_target_fpir.eq(DISPLAY_TARGET_FPIR)
                        & view.diagnostic_fpir.eq(DISPLAY_DIAGNOSTIC_FPIR)]
        display(view.groupby(["compression_profile", "method"]).agg(
            split_count=("partition_seed", "count"), pq_gain_median=("pq_gain", "median"),
            origin_gain_median=("origin_gain", "median"), interaction_median=("interaction", "median")))
        print("채팅용 요약:", result["chat_dir"])
        print("전체 완료 후 상세 ZIP:", result["report_dir"])


## 결과 읽기와 해석

- GPT 채팅에는 출력된 `chat/.../START_HERE.md`와 `analysis.zip`을 첨부하세요.
  ZIP 내부의 `README.md`와 `INDEX.csv`에서 조건별 파일을 찾습니다.
  ZIP을 읽을 수 없는 채팅에서는 로컬에서 풀고 필요한 데이터셋/모델 CSV만 첨부하세요.
- seed별 원자료는 로컬 SQLite에 보존합니다. 전체 완료 시 `reports/.../results.zip`에 모든 상세 표를 Parquet로 저장합니다.
  Python에서는 `from research.experiments.origin_pq_storage import read_report` 후
  `tables, manifest = read_report(명시적인_report_폴더)`로 읽을 수 있습니다.
- 요약 ZIP은 전체 조건을 사용하며, 조건별 CSV는 논리적 페이지로 나뉩니다. 표시 자릿수로 목표 충족을 재판정하지 않습니다.
- 같은 test를 공유하는 20개 seed는 독립 반복이 아닙니다. min/median/max와 CI 끝점 범위는 통합 신뢰구간이 아닙니다.
- 같은 목표 FPIR 비교와 같은 실제 FPIR의 보간 진단을 구분합니다. 진단으로 배포 threshold를 선택하지 않습니다.
- TPIR@20은 정답 Top-20 포함과 정답 점수 threshold 통과를 모두 요구합니다.
- 경험적 결과를 formal/group FPIR 보장으로 표현하지 않습니다. 학습 데이터 overlap 미검증 등 기존 한계는 유지합니다.


### BLUFR 공개 분할 기반의 독립 보정

공개 train/test·test gallery 1,000명·probe 목록을 그대로 사용합니다. train 1,500명 내부에서
압축 학습:보정 = 5:5, 6:4, 7:3, 8:2, 4:6, 3:7, 2:8을 비교합니다.
calibration gallery는 고정 100명×1장입니다. 100→1,000명 검색 규모 전이이며, gallery 이미지 선택 방식도 다릅니다.
별도 BLUFR benchmark 평가는 실행하지 않습니다. 기존 benchmark Python 코드는 보존합니다.
설정 변경 후 **Kernel Restart → Run All**. 기존 완료 run은 덮어쓰지 않습니다.

주 실행점입니다. 처음에는 EXECUTE=False로 검사하고 ready=True일 때 True로 실행합니다.
YAML의 10 trials×7비율×4모델×20 fit/safety seeds = 5,600 jobs입니다. 각 job은 Origin+3PQ를 함께 비교합니다.
DATASETS/RUN_MATRIX/REUSE_PQ_MODELS/SOURCE_REPORT_DIR는 새 경로에 적용되지 않습니다. PQ는 개발 인물만으로 새로 학습합니다.
DISPLAY_*는 표시 필터입니다. 같은 설정으로 재실행하면 완료 작업을 재사용합니다.


In [ ]:
if LFW_PROTOCOL_MODE == "matched_calibration":
    from pathlib import Path
    import sys
    PROJECT_ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
                        if (p / "research").is_dir() and (p / ".git").exists())
    if str(PROJECT_ROOT) not in sys.path:
        sys.path.insert(0, str(PROJECT_ROOT))
    from IPython.display import display
    from research.experiments.lfw_protocol_experiments import run_matched_lfw_calibration
    from research.experiments.origin_vs_pq_calibration import OriginPQSettings
    matched_result = run_matched_lfw_calibration(PROJECT_ROOT, run_matrix=RUN_MATRIX,
        models=MODELS, partition_seeds=PARTITION_SEEDS, settings=OriginPQSettings(**FIT_SETTINGS),
        output_root=LFW_MATCHED_OUTPUT_ROOT, execute=EXECUTE, max_new_jobs=MAX_NEW_JOBS_PER_RUN,
        blas_threads=BLAS_THREADS, minimum_available_gb=MIN_AVAILABLE_RAM_GB,
        maximum_process_gb=MAX_PROCESS_RAM_GB, profiles=PQ_PROFILES, fiqa_variant=FIQA_VARIANT)
    display({k: v for k, v in matched_result.items() if k not in ("method_summary", "inventory", "plan")})
    if "inventory" in matched_result:
        display(matched_result["inventory"])
    if "method_summary" in matched_result:
        view = matched_result["method_summary"]
        display(view.loc[view.model.eq(DISPLAY_MODEL) & view.target_fpir.eq(DISPLAY_TARGET_FPIR)])
    elif "plan" in matched_result:
        display(matched_result["plan"])


In [ ]:
if LFW_PROTOCOL_MODE == "blufr_calibration":
    from pathlib import Path
    import sys
    PROJECT_ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
                        if (p / "research").is_dir() and (p / ".git").exists())
    if str(PROJECT_ROOT) not in sys.path:
        sys.path.insert(0, str(PROJECT_ROOT))
    from IPython.display import display
    import yaml
    from research.experiments.lfw_blufr_calibration import run_calibration
    from research.experiments.origin_vs_pq_calibration import OriginPQSettings
    selected_config = yaml.safe_load((PROJECT_ROOT / LFW_CALIBRATION_CONFIG_PATH).read_text(encoding="utf8"))
    if tuple(selected_config["pq_profiles"]) != tuple(PQ_PROFILES) or selected_config["fiqa_variant"] != FIQA_VARIANT:
        raise ValueError("PQ_PROFILES/FIQA_VARIANT를 선택한 YAML과 일치시키세요.")
    blufr_result = run_calibration(PROJECT_ROOT, config_path=LFW_CALIBRATION_CONFIG_PATH,
        models=MODELS, execute=EXECUTE, partition_seeds=PARTITION_SEEDS,
        settings=OriginPQSettings(**FIT_SETTINGS), max_new_jobs=MAX_NEW_JOBS_PER_RUN,
        blas_threads=BLAS_THREADS, minimum_available_gb=MIN_AVAILABLE_RAM_GB,
        maximum_process_gb=MAX_PROCESS_RAM_GB)
    display({k: v for k, v in blufr_result.items() if k not in ("coverage", "inventory", "missing", "method_summary")})
    for name in ("coverage", "inventory", "missing"):
        if name in blufr_result:
            display(blufr_result[name])
    if "method_summary" in blufr_result:
        view = blufr_result["method_summary"]
        display(view.loc[view.dataset_id.eq(DISPLAY_DATASET) & view.model.eq(DISPLAY_MODEL)
            & view.target_fpir.eq(DISPLAY_TARGET_FPIR) & view.trial_id.eq(DISPLAY_TRIAL_ID)
            & view.development_fraction.eq(DISPLAY_DEVELOPMENT_FRACTION)])
